# Submission: blend of mlp_4out_emb_bn and lgbm_building

Report: [ensemble_report.md](../../ensemble_report.md#blend)

In [ ]:
import os
import sys

import numpy as np
import pandas as pd
import torch

sys.path.insert(0, "../../..")  # the ashrae package lives in the homework folder
from ashrae.config import PROCESSED
from ashrae.data import read_processed
from ashrae.features import build_features
from ashrae.modeling import MeterMLP, make_loaders
from ashrae.submission import build_submission, check_submission, to_readings
from ashrae.tracking import Experiment

# Imported after torch: loading LightGBM first crashes the kernel on macOS (both ship an OpenMP runtime).
import lightgbm as lgb

SMOKE = os.environ.get("SMOKE") == "1"
CHUNK_SIZE = 1_000_000  # test rows predicted at once
W_MLP = 0.65  # MLP weight in the blend, from exp01_blend
if SMOKE:
    EXPERIMENT = "ashrae-submission-smoke"
    N_TRAIN = 20_000
    N_TEST = 50_000
    EPOCHS = 1
    LGBM_PARAMS = {"n_estimators": 10, "learning_rate": 0.1, "num_leaves": 63}
    SUBMISSION_PATH = None  # partial test set, so nothing to submit
else:
    EXPERIMENT = "ashrae-submission"
    N_TRAIN = 2_000_000  # 4M didn't help (exp01_blend)
    N_TEST = None  # all 41.7M test rows
    EPOCHS = 20
    LGBM_PARAMS = {"n_estimators": 500, "learning_rate": 0.1, "num_leaves": 63}
    SUBMISSION_PATH = PROCESSED / "submission_blend.csv"

experiment = Experiment(EXPERIMENT, "submission_blend")

## Training rows

In [ ]:
df = read_processed()
n_buildings = int(df.building_id.max()) + 1
BUILDING = pd.CategoricalDtype(range(n_buildings))
train_df = df[~df.anomaly].sample(N_TRAIN, random_state=0)  # all of 2016, anomalies dropped
del df

year_median = train_df.year_built.median()  # reused for the test set
X_train = build_features(train_df, year_median)
y_train = np.log1p(train_df.meter_reading.to_numpy(dtype="float32"))
print(f"{len(train_df):,} training rows, {X_train.shape[1]} features")


def with_building(X, rows):
    return X.assign(building_id=rows.building_id.astype(int).astype(BUILDING).to_numpy())

## Models

In [ ]:
train_loader, _, scaler = make_loaders(train_df, None, X_train, None)
mlp, run_id = experiment.fit(
    lambda: MeterMLP(X_train.shape[1], n_buildings, batch_norm=True), train_loader, run_name=f"mlp_4out_emb_bn | {N_TRAIN:,} rows",
    epochs=EPOCHS, params={"n_train": N_TRAIN}, tags={"model": "mlp_4out_emb_bn"},
)
mlp = mlp.eval()
print(f"mlp_4out_emb_bn: training RMSLE {experiment.history(run_id, 'train_rmsle')[-1]:.3f}")
del train_loader

In [ ]:
# n_jobs=1: multi-threaded LightGBM segfaults once torch has loaded its OpenMP runtime
lgbm = lgb.LGBMRegressor(**LGBM_PARAMS, random_state=0, verbose=-1, n_jobs=1).fit(with_building(X_train, train_df), y_train)
experiment.log_run(f"lgbm_building | {N_TRAIN:,} rows", params={**LGBM_PARAMS, "n_train": N_TRAIN}, metrics={},
                   tags={"model": "lgbm_building"})

## Test predictions

In [ ]:
test_df = read_processed("test_merged.parquet")
if N_TEST:
    test_df = test_df.head(N_TEST)


@torch.no_grad()
def predict_log(rows):
    """Clipped log1p predictions of both models for `rows`, with the same feature columns and scaler as training."""
    X = build_features(rows, year_median).reindex(columns=X_train.columns, fill_value=0)
    batch = {
        "x": torch.tensor(scaler.transform(X), dtype=torch.float32, device=mlp.device),
        "meter": torch.tensor(rows.meter.to_numpy(), dtype=torch.long, device=mlp.device),
        "building": torch.tensor(rows.building_id.to_numpy(), dtype=torch.long, device=mlp.device),
    }
    return mlp(batch).clamp(min=0).cpu().numpy(), np.clip(lgbm.predict(with_building(X, rows)), 0, None)


starts = range(0, len(test_df), CHUNK_SIZE)
chunks = [predict_log(test_df.iloc[start : start + CHUNK_SIZE]) for start in starts]
mlp_log = np.concatenate([c[0] for c in chunks])
lgbm_log = np.concatenate([c[1] for c in chunks])
del chunks
print(f"correlation of the two models' test predictions (log1p): {np.corrcoef(mlp_log, lgbm_log)[0, 1]:.3f}")
print(f"mean log1p prediction: mlp {mlp_log.mean():.3f}, lgbm {lgbm_log.mean():.3f}")

In [ ]:
reading = to_readings(W_MLP * mlp_log + (1 - W_MLP) * lgbm_log, test_df)  # site 0 electricity back in kBTU
submission = build_submission(test_df, reading)
if SUBMISSION_PATH:
    check_submission(submission)
    submission.to_csv(SUBMISSION_PATH, index=False, float_format="%.4f")
    print(f"saved {SUBMISSION_PATH}: {len(submission):,} rows")
submission.meter_reading.describe().round(2).to_frame()

In [ ]:
if SUBMISSION_PATH:  # multithreaded 7-Zip CLI (brew install sevenzip), much faster than py7zr
    !cd "{SUBMISSION_PATH.parent}" && rm -f "{SUBMISSION_PATH.stem}.7z" && 7zz a -mmt=on "{SUBMISSION_PATH.stem}.7z" "{SUBMISSION_PATH.name}"